# 13 — Tactical video view and per-frame offside (Phase 5)

Combines the study's best detector (03, imgsz 1280), ByteTrack tracks and team assignment (08), and per-frame automatic pitch calibration (12). Output: side-by-side MP4 — the frame with team rings and the offside band, and a live top-down minimap.

**Offside is per-frame only** (no pass-moment or temporal logic) and follows Law 11: the line sits at whichever of the ball and the second-last defender is nearer the goal line; an attacker is flagged if in the opponents' half and beyond it. This is an **offside visualiser, not VAR**: feet are box bottom-centres, and only visible players count — a deeper defender off-screen moves the true line.

**Defending side, from the goalkeeper (plan Section 7):** each frame's visible keeper fixes the goal being defended; the defending *team* is decided once per clip as the team whose players sit nearer that goal on average. Stated as an assumption — it fails in attacking-overload frames.

**Calibration accuracy caveat, up front:** notebook 12 measured a median reprojection error of **2.8 m** for automatic calibration on the test split. That is fine for a tactical minimap and far too coarse for an offside decision; the band drawn here is a visualisation of the logic, not a verdict. The flicker figure at the end quantifies how unstable it is.

Clip: "Match de football France-Allemagne - 16 octobre 2018 - Phase de jeu (1)" by Like tears in rain, Wikimedia Commons, **CC BY-SA 4.0** (output shared under the same licence).

In [ ]:
DET_WEIGHTS, DET_IMGSZ = "/content/results/res_yolo11s_1280/best.pt", 1280
KP_WEIGHTS, KP_IMGSZ, KP_CONF = "/content/results/pitch_kp_yolo11s/best.pt", 960, 0.5
EMA = 0.3          # homography smoothing, as in 12
OUT = "/content/results/tactical"

In [ ]:
import sys, json, pathlib, subprocess, collections
sys.path.insert(0, "/content/ballhawk")
import numpy as np, cv2
from ultralytics import YOLO
import ballhawk_video as bv, ballhawk_pitch as bp

out = pathlib.Path(OUT); out.mkdir(parents=True, exist_ok=True)
VIDEO = bv.fetch_clip()
names = [YOLO(DET_WEIGHTS).names[i] for i in range(len(YOLO(DET_WEIGHTS).names))]
PLAYER, KEEPER, REF, BALL = (names.index(n) for n in ("player", "goalkeeper", "referee", "ball"))
dets = bv.track(DET_WEIGHTS, VIDEO, tracker="bytetrack.yaml", imgsz=DET_IMGSZ)
teams, agreement = bv.assign_teams(VIDEO, dets, PLAYER)
print(f"{len(dets)} frames, {len(teams)} player tracks, team agreement {agreement:.3f}")

In [ ]:
# Pass 1: smoothed homography per frame and every object's pitch position.
kp_model = YOLO(KP_WEIGHTS)
H_s, per_frame = None, []
for frame, d in zip(bv.frames(VIDEO), dets):
    fit = bp.frame_homography(kp_model, frame, KP_IMGSZ, KP_CONF)
    if fit is not None:
        H = fit[0] / fit[0][2, 2]
        H_s = H if H_s is None else EMA * H + (1 - EMA) * H_s
    pitch = bp.project(H_s, bp.foot_points(d["xyxy"])) if (H_s is not None and len(d["cls"])) else np.zeros((0, 2))
    per_frame.append({"H": None if fit is None else H_s, "pitch": pitch})
calibrated = np.mean([f["H"] is not None for f in per_frame])

# Defending goal from keepers; defending team decided once per clip (assumption stated above).
keeper_x = [p[0] for f, d in zip(per_frame, dets) for p, c in zip(f["pitch"], d["cls"]) if c == KEEPER and f["H"] is not None]
goal_x = bp.defending_goal_x(float(np.median(keeper_x))) if keeper_x else None
depth = collections.defaultdict(list)
for f, d in zip(per_frame, dets):
    if f["H"] is None or goal_x is None:
        continue
    for p, c, i in zip(f["pitch"], d["cls"], d["id"]):
        if c == PLAYER and int(i) in teams:
            depth[teams[int(i)]].append(abs(p[0] - goal_x))
defending = min(depth, key=lambda t: np.mean(depth[t])) if depth else None
print(f"calibrated {calibrated:.1%} of frames; defending goal x = {goal_x}; defending team = {defending}")

In [ ]:
# Pass 2: per-frame offside and the side-by-side render.
mm_base, to_px = bp.minimap(scale=8)
raw = out / "tactical_raw.mp4"; writer = None
lines, second_last_ids = [], []
TEAM = bv.TEAM_BGR
for frame, d, f in zip(bv.frames(VIDEO), dets, per_frame):
    mm = mm_base.copy(); line_x = None
    if f["H"] is not None and goal_x is not None and defending is not None:
        role = [("ref" if c == REF else "ball" if c == BALL else "keeper" if c == KEEPER
                 else "def" if teams.get(int(i)) == defending else "att" if int(i) in teams else "unk")
                for c, i in zip(d["cls"], d["id"])]
        on_pitch = [(0 <= p[0] <= bp.LENGTH and 0 <= p[1] <= bp.WIDTH) for p in f["pitch"]]
        dx = [p[0] for p, r, ok in zip(f["pitch"], role, on_pitch) if ok and r in ("def", "keeper")]
        ax = [(k, p[0]) for k, (p, r, ok) in enumerate(zip(f["pitch"], role, on_pitch)) if ok and r == "att"]
        bx = [p[0] for p, r, ok in zip(f["pitch"], role, on_pitch) if ok and r == "ball"]
        line_x, flags = bp.offside(dx, [a for _, a in ax], goal_x, bx[0] if bx else None)
        flagged = {k for (k, _), fl in zip(ax, flags) if fl}
        if line_x is not None:
            frame = bp.draw_offside_line(frame, f["H"], line_x)
            cv2.line(mm, to_px(line_x, 0), to_px(line_x, bp.WIDTH), (52, 104, 235), 2)
        for k, (p, r, ok) in enumerate(zip(f["pitch"], role, on_pitch)):
            if not ok:
                continue
            colour = {"def": TEAM[defending], "att": TEAM[1 - defending], "keeper": bv.KEEPER_BGR,
                      "ref": bv.REFEREE_BGR, "ball": (255, 255, 255), "unk": bv.UNKNOWN_BGR}[r]
            cv2.circle(mm, to_px(*p), 5 if r != "ball" else 4, colour, -1, cv2.LINE_AA)
            if k in flagged:
                cv2.circle(mm, to_px(*p), 9, (0, 0, 255), 2, cv2.LINE_AA)
                x1, y1, x2, y2 = d["xyxy"][k].astype(int)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
    lines.append(line_x)
    for box, c, i in zip(d["xyxy"], d["cls"], d["id"]):
        if c == PLAYER:
            x1, y1, x2, y2 = box.astype(int)
            cv2.ellipse(frame, ((x1 + x2) // 2, y2), (max((x2 - x1) // 2, 6), max((x2 - x1) // 6, 2)), 0, -45, 235,
                        TEAM.get(teams.get(int(i)), bv.UNKNOWN_BGR), 2, cv2.LINE_AA)
    left = cv2.resize(frame, (1280, 720)); right = cv2.resize(mm, (int(mm.shape[1] * 720 / mm.shape[0]), 720))
    canvas = np.hstack([left, right])
    cv2.putText(canvas, "offside visualiser, not VAR - auto calibration ~2.8 m median error", (12, 704),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
    if writer is None:
        writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*"mp4v"), bv.video_fps(VIDEO), canvas.shape[1::-1])
    writer.write(canvas)
writer.release()
final = out / "13_tactical_offside.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw), "-c:v", "libx264", "-pix_fmt", "yuv420p",
                "-crf", "23", str(final)], check=True)
raw.unlink()
print(final)

In [ ]:
# Flicker: how often the drawn line jumps by more than 1 m between consecutive frames.
pairs = [(a, b) for a, b in zip(lines, lines[1:]) if a is not None and b is not None]
jumps = [abs(b - a) for a, b in pairs]
stats = {"frames": len(lines), "calibrated_share": float(calibrated),
         "line_drawn_share": float(np.mean([l is not None for l in lines])),
         "median_line_step_m": float(np.median(jumps)) if jumps else None,
         "share_steps_over_1m": float(np.mean([j > 1 for j in jumps])) if jumps else None,
         "defending_goal_x": goal_x, "defending_team": defending, "team_agreement": agreement,
         "clip": bv.CLIP_CREDIT, "detector": DET_WEIGHTS, "keypoints": KP_WEIGHTS, "ema": EMA}
(out / "tactical_stats.json").write_text(json.dumps(stats, indent=2))
print(json.dumps(stats, indent=2))